### Data Preparation

This notebook is to prepare the data as 3 files due to the constraints of the project, this is directly tied to the original research paper that collected this dataset. One of the hotels is a resort hotel and the other is a city hotel so they were divided based on that. 
A country lookup table was created as the third file, this was done for the bias and fairness analysis.

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [3]:
df_raw = pd.read_csv('./data/hotel_bookings.csv')

print(f'dataset shape: {df_raw.shape}')
print(f'columns ({len(df_raw.columns)}): {list(df_raw.columns)}')

dataset shape: (119390, 32)
columns (32): ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']


In [4]:
# split intom city and resort hotel files, like in the paper
df_city = df_raw[df_raw['hotel'] == 'City Hotel'].copy().reset_index(drop=True)
df_resort = df_raw[df_raw['hotel'] == 'Resort Hotel'].copy().reset_index(drop=True)

print(f'City Hotel rows : {len(df_city):,}')
print(f'Resort Hotel rows : {len(df_resort):,}')
print(f'Total : {len(df_city) + len(df_resort):,}')

City Hotel rows : 79,330
Resort Hotel rows : 40,060
Total : 119,390


In [13]:
countries = df_raw['country'].unique()
# print(countries)
countries
# print(f'Countries : \n {", ".join(countries)}')

<ArrowStringArray>
['PRT', 'GBR', 'USA', 'ESP', 'IRL', 'FRA',   nan, 'ROU', 'NOR', 'OMN',
 ...
 'ATA', 'GTM', 'ASM', 'MRT', 'NCL', 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 178, dtype: str

In [ ]:
# ── Create country lookup table (File 3) ─────────────────────────────────────
# Maps ISO 3166-1 alpha-3 country codes to region and continent
# This enriches the dataset with geographic context for bias analysis

country_region_map = {
    'PRT': ('Southern Europe', 'Europe'),
    'GBR': ('Northern Europe', 'Europe'),
    'FRA': ('Western Europe', 'Europe'),
    'ESP': ('Southern Europe', 'Europe'),
    'DEU': ('Western Europe', 'Europe'),
    'ITA': ('Southern Europe', 'Europe'),
    'IRL': ('Northern Europe', 'Europe'),
    'BEL': ('Western Europe', 'Europe'),
    'BRA': ('South America', 'Americas'),
    'NLD': ('Western Europe', 'Europe'),
    'USA': ('North America', 'Americas'),
    'CHN': ('Eastern Asia', 'Asia'),
    'AUT': ('Western Europe', 'Europe'),
    'POL': ('Eastern Europe', 'Europe'),
    'CHE': ('Western Europe', 'Europe'),
    'RUS': ('Eastern Europe', 'Europe'),
    'NOR': ('Northern Europe', 'Europe'),
    'SWE': ('Northern Europe', 'Europe'),
    'DNK': ('Northern Europe', 'Europe'),
    'MOZ': ('Sub-Saharan Africa', 'Africa'),
    'AGO': ('Sub-Saharan Africa', 'Africa'),
    'CN':  ('Eastern Asia', 'Asia'),
    'NULL': ('Unknown', 'Unknown'),
}

# Get all countries in the dataset and build lookup
all_countries = df_raw['country'].dropna().unique()
lookup_rows = []
for c in all_countries:
    region, continent = country_region_map.get(c, ('Other', 'Other'))
    lookup_rows.append({'country_code': c, 'region': region, 'continent': continent})

df_country_lookup = pd.DataFrame(lookup_rows).sort_values('country_code').reset_index(drop=True)

print(f'Country lookup table: {df_country_lookup.shape}')
df_country_lookup.head(10)

In [ ]:
# create a country lookup table for geographic context for bias analysis
# country codes are expected in ISO 3166-1 alpha-3 format
iso_regions_url = 'https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv'

df_iso_regions = pd.read_csv(iso_regions_url, usecols=['alpha-3', 'region', 'sub-region'])
country_region_map = {
    row['alpha-3']: (row['sub-region'], row['region'])
    for _, row in df_iso_regions.iterrows()
}

# normalize known non-standard/legacy values observed in this dataset
country_aliases = {
    'CN': 'CHN',   # alpha-2 -> alpha-3
    'TMP': 'TLS',  # legacy Timor-Leste code
}

# explicit placeholders
country_region_map['NULL'] = ('Unknown', 'Unknown')

# get all countries in the dataset and build lookup
all_countries = df_raw['country'].dropna().unique()
lookup_rows = []
for c in all_countries:
    c_norm = country_aliases.get(c, c)
    region, continent = country_region_map.get(c_norm, ('Other', 'Other'))
    lookup_rows.append({'country_code': c, 'country_code_iso3': c_norm, 'region': region, 'continent': continent})

df_country_lookup = pd.DataFrame(lookup_rows).sort_values('country_code').reset_index(drop=True)

print(f'Country lookup table: {df_country_lookup.shape}')
df_country_lookup.head(10)

Country lookup table: (177, 4)


,country_code,country_code_iso3,region,continent
0,ABW,ABW,Latin America and the Caribbean,Americas
1,AGO,AGO,Sub-Saharan Africa,Africa
2,AIA,AIA,Latin America and the Caribbean,Americas
3,ALB,ALB,Southern Europe,Europe
4,AND,AND,Southern Europe,Europe
5,ARE,ARE,Western Asia,Asia
6,ARG,ARG,Latin America and the Caribbean,Americas
7,ARM,ARM,Western Asia,Asia
8,ASM,ASM,Polynesia,Oceania
9,ATA,ATA,NaN,NaN


In [5]:
# save the files 
df_city.to_csv('./data/city_hotel.csv', index=False)
df_resort.to_csv('./data/resort_hotel.csv', index=False)
df_country_lookup.to_csv('./data/country_lookup.csv', index=False)